# 03 — Baseline Services Exploration

Deep-dive into the four non-LLM translation sources, grouped by epistemic stance:

- **Community-curated reference** (1 source): **Wikipedia** interlanguage links — what scholars and editors in each language community have collectively decided this concept is called. Not machine translation; a human-negotiated equivalence mapping. The category is extensible — future additions could include IATE, LCSH, DH journal subject headings, or Wikidata.
- **Machine translation baselines** (3 sources): **Google Translate**, **EasyNMT** (Helsinki-NLP opus-mt), and **Lingvanex** — algorithmic translations produced by neural MT systems trained on parallel corpora.

These two categories answer different questions, and conflating them obscures what the LLM comparison is really testing. The MT baselines tell us *how well current statistical MT handles DH terminology*. Wikipedia tells us *what humans in each language community have already decided to call this concept*. LLM outputs sit between these two poles.

Because these sources run once per language with no prompt variation, they are a cleaner lens for several distinct questions:

- **Coverage**: which language families can (and can't) each source reach, and why?
- **Overlap**: how many of the 880 languages have translations from multiple sources, and how many have none at all?
- **Fidelity**: when a source does produce a translation, does it produce a real translation or echo "Digital Humanities" unchanged?
- **Quality**: how do the automated quality flags from `translation_classifier.py` compare across sources?
- **Tier funnel**: what counts as usable data after applying quality filters at each exclusion tier?

**Sections**
1. Coverage by source and language family — including EasyNMT structural gaps and overlap
2. Pass-through rate: sources that echo the source term unchanged
3. Automated quality flags per source
4. Mixed-script output in baseline translations
5. Tier exclusion view: what remains after applying quality filters


In [1]:
import os
import sys
from collections import defaultdict

import pandas as pd
import altair as alt
from pathlib import Path

alt.data_transformers.enable("vegafusion")

sys.path.insert(0, str(Path("..").resolve()))
from scripts.utils import get_data_directory_path, read_csv_file, get_language_family
from scripts.exploration.explore_confidence_within_variant import load_variant_df
from scripts.exploration.translation_classifier import (
    curate_translation,
    has_source_leakage,
    is_placeholder_term,
    is_repetition_loop,
    has_extreme_term_length,
    has_unicode_escape,
)

DATA_DIR = get_data_directory_path()
TERM = "Digital Humanities"
TERM_SLUG = TERM.lower().replace(" ", "_")

BASELINE_SERVICES = {
    "Wikipedia": "wikipedia_translated_term",
    "Google Translate": "gt_translated_term",
    "EasyNMT": "enmt_translated_term",
    "Lingvanex": "lingvanex_translated_term",
}
LLM_SERVICES = {
    "OpenAI":   "openai_translated_term",
    "Claude":   "claude_translated_term",
    "Gemini":   "gemini_translated_term",
    "DeepSeek": "deepseek_translated_term",
    "Llama":    "llama_translated_term",
    "Gemma":    "gemma_translated_term",
    "Qwen":     "qwen_translated_term",
    "Mistral":  "mistral_translated_term",
}
ALL_SERVICES = {**BASELINE_SERVICES, **LLM_SERVICES}

print(f"Data directory: {DATA_DIR}")

Retrieving translation pipeline data directory path...

Data directory: /Users/zleblanc/CodingDH/translation_transmogrification_pipeline/datasets


In [2]:
# ── Manual exclusions ───────────────────────────────────────────────────────
from scripts.utils import load_manual_exclusions

_excl_eval_dir = os.path.join(DATA_DIR, "translated_terms", TERM_SLUG, "evaluation")
analysis_langs, search_terms, corrections = load_manual_exclusions(_excl_eval_dir)
print(f"Manual exclusions loaded:")
print(f"  analysis_exclusion : {len(analysis_langs)} language codes  (dropped from all analysis)")
print(f"  search_exclusion   : {len(search_terms)} (language, term) pairs  (excluded from search only)")
print(f"  term_correction    : {len(corrections)} corrections")

Manual exclusions loaded:
  analysis_exclusion : 95 language codes  (dropped from all analysis)
  search_exclusion   : 381 (language, term) pairs  (excluded from search only)
  term_correction    : 93 corrections


In [3]:
# Load the minimal-variant merged dataframe — baseline columns are identical across variants
df = load_variant_df(DATA_DIR, TERM_SLUG, "minimal")
df["language_family"] = df["language_code"].apply(get_language_family)

total_langs = df["language_code"].nunique()
print(f"{TERM}: {total_langs} languages loaded")

# Quality flags from notebook 02
flags_path = os.path.join(DATA_DIR, "translated_terms", TERM_SLUG, "evaluation", "quality_flags.csv")
flags = read_csv_file(flags_path)
print(f"Quality flags: {len(flags)} rows, {flags.columns.tolist()}")

# Drop analysis-excluded languages
df = df[~df["language_code"].isin(analysis_langs)].reset_index(drop=True)
print(f"After dropping {len(analysis_langs)} analysis-excluded languages: {df['language_code'].nunique()} languages remain.")

Retrieving translation pipeline data directory path...

Digital Humanities: 880 languages loaded
Quality flags: 880 rows, ['language_code', 'language_name', 'language_family', 'has_missing_rationale', 'missing_rationale_services', 'has_mixed_script', 'mixed_script_services', 'has_romanization', 'romanization_services', 'has_script_disagreement', 'script_disagr_services', 'has_source_term', 'source_term_services', 'has_placeholder_term', 'placeholder_term_services', 'has_repetition_loop', 'repetition_loop_services', 'has_extreme_term_length', 'extreme_term_length_services', 'has_unicode_escape', 'unicode_escape_services', 'has_short_translation', 'short_translation_services', 'has_any_mixing', 'any_mixing_services', 'has_refusal_rationale', 'refusal_rationale_services', 'has_transliteration_rationale', 'transliteration_rationale_services', 'has_placeholder_rationale', 'placeholder_rationale_services', 'has_language_name_term', 'language_name_term_services', 'has_unexpected_rationale_language', 'unexpected_rationale_language_services', 'quali

## 3.1 — Coverage by Service and Language Family

How many of the 880 languages in the pipeline does each baseline service reach? Baseline services don't use prompt variants, so this is a single clean count per language.


In [4]:
# Overall coverage counts
coverage_rows = []
for svc, col in BASELINE_SERVICES.items():
    if col not in df.columns:
        continue
    n = df[col].notna().sum()
    coverage_rows.append({
        "Service": svc,
        "Covered": int(n),
        "Missing": int(total_langs - n),
        "Coverage %": round(100 * n / total_langs, 1),
    })

coverage_df = pd.DataFrame(coverage_rows).sort_values("Covered", ascending=False)
display(coverage_df)

,Service,Covered,Missing,Coverage %
1,Google Translate,229,651,26.0
3,Lingvanex,100,780,11.4
2,EasyNMT,92,788,10.5
0,Wikipedia,35,845,4.0


In [5]:
# Coverage bar chart
bar = alt.Chart(coverage_df).mark_bar().encode(
    x=alt.X("Covered:Q", title="Languages with a translation"),
    y=alt.Y("Service:N", sort="-x", title=None),
    color=alt.Color("Service:N", legend=None),
    tooltip=["Service", "Covered", "Coverage %"],
).properties(title="Baseline service coverage", width=500, height=150)

rule = alt.Chart(pd.DataFrame({"x": [total_langs]})).mark_rule(
    color="grey", strokeDash=[4, 4]
).encode(x="x:Q")

(bar + rule)

alt.LayerChart(...)

In [6]:
# Coverage heatmap: service × language family
fam_rows = []
for svc, col in BASELINE_SERVICES.items():
    if col not in df.columns:
        continue
    grp = df.groupby("language_family")[col].agg(
        total="count",
        covered=lambda x: x.notna().sum(),
    ).reset_index()
    grp["Service"] = svc
    grp["pct"] = grp["covered"] / grp["total"]
    fam_rows.append(grp)

fam_df = pd.concat(fam_rows, ignore_index=True)

# Sort families by total language count (descending)
fam_totals = df.groupby("language_family")["language_code"].nunique().sort_values(ascending=False)
fam_order = fam_totals.index.tolist()

heatmap = alt.Chart(fam_df).mark_rect().encode(
    x=alt.X("Service:N", title=None),
    y=alt.Y("language_family:N", sort=fam_order, title=None),
    color=alt.Color(
        "pct:Q",
        scale=alt.Scale(scheme="greens", domain=[0, 1]),
        title="Coverage fraction",
    ),
    tooltip=[
        alt.Tooltip("Service:N"),
        alt.Tooltip("language_family:N", title="Family"),
        alt.Tooltip("covered:Q", title="Covered"),
        alt.Tooltip("total:Q", title="Total in family"),
        alt.Tooltip("pct:Q", title="Coverage", format=".0%"),
    ],
).properties(
    title="Baseline coverage by language family",
    width=300,
    height=500,
)
heatmap

alt.Chart(...)

### EasyNMT Structural Gaps by Family

EasyNMT's low overall coverage (100/880, 11.4%) is not a random scatter of failures — it reflects the availability of Helsinki NLP opus-mt models for each language pair. When no opus-mt model exists for a source→target language pair, EasyNMT returns a 404 and leaves the field empty. This means its gaps are *structural*: entire language families are out of scope, not just individual languages.

Thirteen families have **zero** EasyNMT coverage (including Indigenous North American, Tai, Caucasian, Eskimo-Aleut, and all South American indigenous languages). For the families that are covered, rates rarely exceed 30%, with Niger-Kordofanian (19%) and Austronesian (23%) being the highest. Indo-European — the family with the most opus-mt models — still reaches only 11% because the pipeline includes many lower-resource IE languages outside the standard MT training distribution.


In [7]:
# EasyNMT structural gaps by language family
enmt_col = BASELINE_SERVICES["EasyNMT"]

fam_enmt = (
    df.groupby("language_family")
    .agg(
        n_langs=("language_code", "nunique"),
        n_covered=(enmt_col, lambda x: x.notna().sum()),
    )
    .reset_index()
)
fam_enmt["coverage_rate"] = fam_enmt["n_covered"] / fam_enmt["n_langs"]
fam_enmt["zero_coverage"] = fam_enmt["n_covered"] == 0
fam_enmt = fam_enmt.sort_values("coverage_rate")

enmt_bar = alt.Chart(fam_enmt).mark_bar().encode(
    y=alt.Y("language_family:N", sort="x", title=None),
    x=alt.X("coverage_rate:Q", axis=alt.Axis(format="%"), title="EasyNMT coverage rate"),
    color=alt.condition(
        alt.datum.zero_coverage,
        alt.value("#d62728"),
        alt.value("#1f77b4"),
    ),
    tooltip=[
        "language_family:N",
        alt.Tooltip("coverage_rate:Q", format=".1%"),
        alt.Tooltip("n_covered:Q", title="languages covered"),
        alt.Tooltip("n_langs:Q", title="total languages"),
    ],
).properties(width=420, height=400,
             title="EasyNMT coverage by family — red = zero (no opus-mt model for this family)")
display(enmt_bar)

zero_fams = fam_enmt[fam_enmt["zero_coverage"]]["language_family"].tolist()
print(f"Families with zero EasyNMT coverage ({len(zero_fams)}):")
for f in zero_fams:
    n = int(fam_enmt.loc[fam_enmt["language_family"]==f, "n_langs"].iloc[0])
    print(f"  {f} ({n} languages)")


alt.Chart(...)

Families with zero EasyNMT coverage (14):
  Dravidian languages (8 languages)
  Japonic languages (1 languages)
  Hmong-Mien languages (4 languages)
  Eskimo-Aleut languages (5 languages)
  Tai-Kadai languages (8 languages)
  Chukotko-Kamchatkan languages (1 languages)
  Central American Indian languages (4 languages)
  Khoisan languages (1 languages)
  Caucasian languages (15 languages)
  Sign languages (1 languages)
  Australian languages (1 languages)
  South American Indian languages (11 languages)
  North American Indian languages (49 languages)
  Language isolate (8 languages)


### Service Overlap: How Many Languages Have Baseline Coverage?

Of the 880 languages in the pipeline, **631 (72%)** have no translation from any of the four baseline services. This is the starting gap that LLM services fill. The overlap distribution shows how many languages have 1, 2, 3, or all 4 baseline translations, giving a sense of how much cross-service validation is available at the baseline level.


In [8]:
# Service overlap: how many baseline services cover each language?
df["n_baseline"] = sum(
    df[col].notna().astype(int)
    for col in BASELINE_SERVICES.values()
    if col in df.columns
)

overlap_counts = df["n_baseline"].value_counts().sort_index().reset_index()
overlap_counts.columns = ["n_services", "n_languages"]
overlap_counts["label"] = overlap_counts["n_services"].map({
    0: "No baseline coverage",
    1: "1 service",
    2: "2 services",
    3: "3 services",
    4: "All 4 services",
})

print("Baseline service overlap across 880 languages:")
for _, r in overlap_counts.iterrows():
    pct = r["n_languages"] / total_langs * 100
    print(f"  {r['label']:25s}: {r['n_languages']:4d}  ({pct:.1f}%)")

overlap_bar = alt.Chart(overlap_counts).mark_bar().encode(
    x=alt.X("label:N",
            sort=["No baseline coverage","1 service","2 services","3 services","All 4 services"],
            title=None),
    y=alt.Y("n_languages:Q", title="languages"),
    color=alt.Color("n_services:O", scale=alt.Scale(scheme="blues"), legend=None),
    tooltip=["label:N", "n_languages:Q"],
).properties(width=340, height=200, title="How many baseline services cover each language?")
display(overlap_bar)

zero_by_fam = (
    df[df["n_baseline"] == 0]
    .groupby("language_family")["language_code"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="n_zero")
)
print(f"\nZero-baseline-coverage languages by family:")
for _, r in zero_by_fam.iterrows():
    total_fam = df[df["language_family"] == r["language_family"]]["language_code"].nunique()
    pct = r["n_zero"] / total_fam * 100
    print(f"  {r['language_family']:45s}: {r['n_zero']:3d}/{total_fam:3d} ({pct:.0f}%)")


Baseline service overlap across 880 languages:
  No baseline coverage     :  543  (61.7%)
  1 service                :  107  (12.2%)
  2 services               :   78  (8.9%)
  3 services               :   35  (4.0%)
  All 4 services           :   22  (2.5%)


alt.Chart(...)


Zero-baseline-coverage languages by family:
  Indo-European languages                      : 142/227 (63%)
  Niger-Kordofanian languages                  : 101/145 (70%)
  North American Indian languages              :  47/ 49 (96%)
  Austronesian languages                       :  42/ 78 (54%)
  Sino-Tibetan languages                       :  39/ 47 (83%)
  Afro-Asiatic languages                       :  37/ 44 (84%)
  Nilo-Saharan languages                       :  20/ 26 (77%)
  Uralic languages                             :  19/ 26 (73%)
  Altaic languages                             :  15/ 30 (50%)
  Artificial languages                         :  13/ 14 (93%)
  Caucasian languages                          :  11/ 15 (73%)
  Austro-Asiatic languages                     :   9/ 12 (75%)
  Creoles and pidgins                          :   8/ 15 (53%)
  South American Indian languages              :   8/ 11 (73%)
  Language isolate                             :   7/  8 (88%)
  Tai-Kada

## 3.2 — Pass-Through Rate: Services that Echo the Source Term

A translation that returns "Digital Humanities" unchanged is not really a translation; it is a coverage failure that looks like data. The rate at which each service does this tells us how much of the nominal coverage is genuine.

In [9]:
pass_rows = []
for svc, col in BASELINE_SERVICES.items():
    if col not in df.columns:
        continue
    vals = df[col].dropna().astype(str)
    covered = len(vals)
    exact = (vals.str.strip().str.lower() == TERM.lower()).sum()
    contains = vals.str.contains(TERM, case=False, na=False).sum()
    source_leak = sum(has_source_leakage(v, TERM) for v in vals)
    pass_rows.append({
        "Service": svc,
        "Covered": covered,
        "Exact pass-through": int(exact),
        "Contains source term": int(contains),
        "Source leakage (incl. initials)": int(source_leak),
        "Pass-through %": round(100 * exact / covered, 1) if covered else 0,
        "Source leakage %": round(100 * source_leak / covered, 1) if covered else 0,
    })

pass_df = pd.DataFrame(pass_rows)
display(pass_df)

,Service,Covered,Exact pass-through,Contains source term,Source leakage (incl. initials),Pass-through %,Source leakage %
0,Wikipedia,35,2,2,2,5.7,5.7
1,Google Translate,229,14,17,17,6.1,7.4
2,EasyNMT,92,1,1,1,1.1,1.1
3,Lingvanex,100,17,17,17,17.0,17.0


In [10]:
# Stacked bar: genuine translation vs pass-through vs source-leakage
stacked_rows = []
for _, row in pass_df.iterrows():
    exact = row["Exact pass-through"]
    leak_extra = row["Source leakage (incl. initials)"] - exact
    genuine = row["Covered"] - row["Source leakage (incl. initials)"]
    stacked_rows += [
        {"Service": row["Service"], "Category": "Genuine translation", "Count": genuine},
        {"Service": row["Service"], "Category": "Contains DH (non-exact)", "Count": max(0, leak_extra)},
        {"Service": row["Service"], "Category": "Exact pass-through", "Count": exact},
    ]

stacked_df = pd.DataFrame(stacked_rows)

stacked_bar = alt.Chart(stacked_df).mark_bar().encode(
    x=alt.X("sum(Count):Q", title="Languages"),
    y=alt.Y("Service:N", sort="-x", title=None),
    color=alt.Color(
        "Category:N",
        scale=alt.Scale(
            domain=["Genuine translation", "Contains DH (non-exact)", "Exact pass-through"],
            range=["#4c9b5e", "#f0a830", "#c9413a"],
        ),
    ),
    tooltip=["Service", "Category", "sum(Count):Q"],
    order=alt.Order("Category:N", sort="ascending"),
).properties(title="Baseline coverage breakdown: genuine vs pass-through", width=500, height=150)

stacked_bar

alt.Chart(...)

In [11]:
# Which languages are pass-throughs? Show the top offenders per service
for svc, col in BASELINE_SERVICES.items():
    if col not in df.columns:
        continue
    passthrough_mask = df[col].str.strip().str.lower() == TERM.lower()
    rows = df.loc[passthrough_mask, ["language_code", "language_name", "language_family", col]]
    if len(rows):
        print(f"\n{svc} — {len(rows)} exact pass-throughs:")
        display(rows.reset_index(drop=True))
    else:
        print(f"\n{svc} — no exact pass-throughs")


Wikipedia — 2 exact pass-throughs:


,language_code,language_name,language_family,wikipedia_translated_term
0,de,German,Indo-European languages,Digital Humanities
1,en,English,Indo-European languages,digital humanities



Google Translate — 14 exact pass-throughs:


,language_code,language_name,language_family,gt_translated_term
0,pag,Pangasinan,Austronesian languages,Digital Humanities
1,pam,Kapampangan,Austronesian languages,Digital Humanities
2,mh,Marshallese,Austronesian languages,Digital Humanities
3,tl,Tagalog,Austronesian languages,Digital Humanities
4,sus,Susu,Niger-Kordofanian languages,Digital Humanities
5,ch,Chamorro,Austronesian languages,Digital Humanities
6,fil,Filipino,Austronesian languages,Digital Humanities
7,ach,Acoli,Nilo-Saharan languages,Digital Humanities
8,bik,Bikol,Austronesian languages,Digital Humanities
9,bcl,Bikol,Austronesian languages,Digital Humanities



EasyNMT — 1 exact pass-throughs:


,language_code,language_name,language_family,enmt_translated_term
0,it,Italian,Indo-European languages,Digital Humanities



Lingvanex — 17 exact pass-throughs:


,language_code,language_name,language_family,lingvanex_translated_term
0,ny,Chichewa; Chewa; Nyanja,Niger-Kordofanian languages,Digital Humanities
1,no,Norwegian,Indo-European languages,Digital Humanities
2,nl,Dutch,Indo-European languages,Digital Humanities
3,mg,Malagasy,Austronesian languages,Digital Humanities
4,lo,Lao,Tai-Kadai languages,Digital Humanities
5,yo,Yoruba,Niger-Kordofanian languages,Digital Humanities
6,sn,Shona,Niger-Kordofanian languages,Digital Humanities
7,tl,Tagalog,Austronesian languages,Digital Humanities
8,st,Southern Sotho,Niger-Kordofanian languages,Digital Humanities
9,bs,Bosnian,Indo-European languages,Digital Humanities


## 3.3 — Automated Quality Flags per Baseline Service

Run each baseline translation through the full `translation_classifier.py` quality check suite. Baseline services don't produce rationale text or have prompt variants, so `has_missing_rationale` and `has_script_disagreement` don't apply here. The relevant flags are the translation-level ones.

In [12]:
CHECKERS = {
    "placeholder": is_placeholder_term,
    "repetition_loop": is_repetition_loop,
    "extreme_length": has_extreme_term_length,
    "unicode_escape": has_unicode_escape,
}

flag_rows = []
for svc, col in BASELINE_SERVICES.items():
    if col not in df.columns:
        continue
    vals = df[col].dropna().astype(str)
    covered = len(vals)
    row = {"Service": svc, "Covered": covered}
    for flag_name, fn in CHECKERS.items():
        count = sum(fn(v) for v in vals)
        row[flag_name] = count
        row[f"{flag_name}_pct"] = round(100 * count / covered, 1) if covered else 0

    # mixed-script / stripped via curate_translation
    nulled = stripped = 0
    for v in vals:
        _, action = curate_translation(v)
        if action == "nulled":
            nulled += 1
        elif action == "stripped":
            stripped += 1
    row["mixed_script"] = nulled
    row["mixed_script_pct"] = round(100 * nulled / covered, 1) if covered else 0
    row["romanization_stripped"] = stripped
    row["romanization_stripped_pct"] = round(100 * stripped / covered, 1) if covered else 0

    flag_rows.append(row)

flag_df = pd.DataFrame(flag_rows)

# Display count table
count_cols = ["Service", "Covered", "placeholder", "repetition_loop", "extreme_length",
              "unicode_escape", "mixed_script", "romanization_stripped"]
display(flag_df[count_cols])

,Service,Covered,placeholder,repetition_loop,extreme_length,unicode_escape,mixed_script,romanization_stripped
0,Wikipedia,35,0,0,0,0,0,0
1,Google Translate,229,0,0,0,0,0,0
2,EasyNMT,92,0,1,2,0,0,0
3,Lingvanex,100,0,0,0,0,0,0


In [13]:
# Flag rate chart
flag_long_rows = []
flag_display = {
    "placeholder": "Placeholder/refusal",
    "repetition_loop": "Repetition loop",
    "extreme_length": "Extreme length (>100 chars)",
    "unicode_escape": "Unicode escape (\\uXXXX)",
    "mixed_script": "Mixed script (nulled)",
    "romanization_stripped": "Romanization stripped",
}
for _, row in flag_df.iterrows():
    for flag, label in flag_display.items():
        pct_col = f"{flag}_pct"
        if pct_col in flag_df.columns:
            flag_long_rows.append({
                "Service": row["Service"],
                "Flag": label,
                "Rate": row[pct_col],
                "Count": int(row[flag]),
            })

flag_long_df = pd.DataFrame(flag_long_rows)

# Only show flags that actually fired
nonzero_flags = flag_long_df.groupby("Flag")["Count"].sum()
active_flags = nonzero_flags[nonzero_flags > 0].index.tolist()

if active_flags:
    active_df = flag_long_df[flag_long_df["Flag"].isin(active_flags)]
    flag_chart = alt.Chart(active_df).mark_bar().encode(
        x=alt.X("Rate:Q", title="% of covered translations"),
        y=alt.Y("Service:N", title=None),
        color=alt.Color("Service:N", legend=None),
        row=alt.Row("Flag:N", title=None),
        tooltip=["Service", "Flag", "Count", alt.Tooltip("Rate:Q", format=".1f", title="%")],
    ).properties(width=400, height=80, title="Baseline quality flag rates (% of covered languages)")
    display(flag_chart)
else:
    print("No quality flags fired for any baseline service.")

alt.Chart(...)

In [14]:
# Show offending rows for any flagged baseline translations
for svc, col in BASELINE_SERVICES.items():
    if col not in df.columns:
        continue
    rows_out = []
    for _, row in df[["language_code", "language_name", "language_family", col]].dropna(subset=[col]).iterrows():
        v = str(row[col])
        flags_fired = []
        for flag_name, fn in CHECKERS.items():
            if fn(v):
                flags_fired.append(flag_name)
        _, action = curate_translation(v)
        if action in ("nulled", "stripped"):
            flags_fired.append(action)
        if flags_fired:
            rows_out.append({
                "language_code": row["language_code"],
                "language_name": row["language_name"],
                "language_family": row["language_family"],
                "translation": v[:120],
                "flags": ", ".join(flags_fired),
            })
    if rows_out:
        print(f"\n{svc} — {len(rows_out)} flagged translations:")
        display(pd.DataFrame(rows_out))
    else:
        print(f"\n{svc} — no quality flags")


Wikipedia — no quality flags

Google Translate — no quality flags

EasyNMT — 3 flagged translations:


,language_code,language_name,language_family,translation,flags
0,loz,Lozi,Niger-Kordofanian languages,Litaba za Kwaikale ze Bulezwi Mwa Bibele ka za...,extreme_length
1,vi,Vietnamese,Austro-Asiatic languages,Hệ bình bình bình bình bình bình bình bình bìn...,repetition_loop
2,bg,Bulgarian,Indo-European languages,(Средредредредредредредредредредредредредредре...,extreme_length



Lingvanex — no quality flags


## 3.4 — Mixed-Script Output

Do any baseline services produce mixed-script output (characters from two or more writing systems interleaved in a single term)? This is relatively rare in MT systems compared to LLMs, but EasyNMT's open-domain opus-mt models can produce unexpected romanization helpers.

In [15]:
from scripts.utils import detect_dominant_script

script_rows = []
for svc, col in BASELINE_SERVICES.items():
    if col not in df.columns:
        continue
    for _, row in df[["language_code", "language_name", "language_family", col]].dropna(subset=[col]).iterrows():
        v = str(row[col])
        script = detect_dominant_script(v)
        _, action = curate_translation(v)
        script_rows.append({
            "Service": svc,
            "language_code": row["language_code"],
            "language_name": row["language_name"],
            "language_family": row["language_family"],
            "translation": v[:80],
            "dominant_script": script,
            "clean_action": action,
        })

script_df = pd.DataFrame(script_rows)

# Script distribution per service
script_dist = (
    script_df.groupby(["Service", "dominant_script"])
    .size()
    .reset_index(name="count")
)
script_chart = alt.Chart(script_dist).mark_bar().encode(
    x=alt.X("count:Q", title="Translations"),
    y=alt.Y("dominant_script:N", title=None, sort="-x"),
    color=alt.Color("Service:N"),
    tooltip=["Service", "dominant_script", "count"],
).properties(
    title="Script distribution of baseline translations",
    width=400, height=250
)
script_chart

alt.Chart(...)

In [16]:
# Translations that needed cleaning (stripped or nulled)
needing_clean = script_df[script_df["clean_action"].isin(["stripped", "nulled"])]
if len(needing_clean):
    print(f"{len(needing_clean)} baseline translations needed cleaning:")
    display(needing_clean[["Service", "language_name", "translation", "clean_action"]].reset_index(drop=True))
else:
    print("No baseline translations required cleaning — all pass through unchanged or as-is.")

No baseline translations required cleaning — all pass through unchanged or as-is.


## 3.5 — Tier 1 Exclusion View: Quality Filters Applied

Under the **Tier 1 exclusion policy** (service exploration), the only automated filters applied to baseline services are the translation-level quality flags — mixed script (nulled), unicode escapes, extreme length, repetition loops, and placeholder/refusal terms.

Pass-throughs (exact "Digital Humanities" echoes) are **not** automatically excluded at Tier 1 because the fact that a service has no translation is itself meaningful data. They are flagged via `has_source_term` in `quality_flags.csv` and excluded at Tiers 2 and 3.

In [17]:
# Tier 1: count usable translations after removing translation-error flags
tier1_rows = []
for svc, col in BASELINE_SERVICES.items():
    if col not in df.columns:
        continue
    vals = df[col].dropna().astype(str)
    covered = len(vals)
    usable = 0
    excluded = 0
    for v in vals:
        _, action = curate_translation(v)
        is_bad = (
            action in ("nulled", "placeholder")
            or is_repetition_loop(v)
            or has_extreme_term_length(v)
            or has_unicode_escape(v)
        )
        if is_bad:
            excluded += 1
        else:
            usable += 1
    tier1_rows.append({
        "Service": svc,
        "Covered": covered,
        "Excluded (Tier 1 filters)": excluded,
        "Usable": usable,
        "Usable %": round(100 * usable / covered, 1) if covered else 0,
    })

tier1_df = pd.DataFrame(tier1_rows)
display(tier1_df)

,Service,Covered,Excluded (Tier 1 filters),Usable,Usable %
0,Wikipedia,35,0,35,100.0
1,Google Translate,229,0,229,100.0
2,EasyNMT,92,3,89,96.7
3,Lingvanex,100,0,100,100.0


In [18]:
# Tier 2: additionally exclude pass-throughs (source term in translation)
tier2_rows = []
for svc, col in BASELINE_SERVICES.items():
    if col not in df.columns:
        continue
    vals = df[col].dropna().astype(str)
    covered = len(vals)
    usable = 0
    excluded = 0
    for v in vals:
        _, action = curate_translation(v)
        is_bad = (
            action in ("nulled", "placeholder")
            or is_repetition_loop(v)
            or has_extreme_term_length(v)
            or has_unicode_escape(v)
            or has_source_leakage(v, TERM)
        )
        if is_bad:
            excluded += 1
        else:
            usable += 1
    tier2_rows.append({
        "Service": svc,
        "Covered": covered,
        "Excluded (Tier 1 + source leakage)": excluded,
        "Usable": usable,
        "Usable %": round(100 * usable / covered, 1) if covered else 0,
    })

tier2_df = pd.DataFrame(tier2_rows)
print("Tier 2 (Tier 1 filters + source leakage exclusion):")
display(tier2_df)

Tier 2 (Tier 1 filters + source leakage exclusion):


,Service,Covered,Excluded (Tier 1 + source leakage),Usable,Usable %
0,Wikipedia,35,2,33,94.3
1,Google Translate,229,17,212,92.6
2,EasyNMT,92,4,88,95.7
3,Lingvanex,100,17,83,83.0


In [20]:
# Comparison: coverage → Tier 1 usable → Tier 2 usable
compare_rows = []
for (_, r1), (_, r2) in zip(tier1_df.iterrows(), tier2_df.iterrows()):
    svc = r1["Service"]
    compare_rows += [
        {"Service": svc, "Stage": "Nominal coverage", "Count": r1["Covered"]},
        {"Service": svc, "Stage": "After Tier 1 filters", "Count": r1["Usable"]},
        {"Service": svc, "Stage": "After Tier 2 filters", "Count": r2["Usable"]},
    ]

compare_df = pd.DataFrame(compare_rows)
stage_order = ["Nominal coverage", "After Tier 1 filters", "After Tier 2 filters"]

funnel = alt.Chart(compare_df).mark_bar().encode(
    x=alt.X("Count:Q", title="Languages"),
    y=alt.Y("Stage:N", sort=stage_order, title=None),
    color=alt.Color(
        "Stage:N",
        sort=stage_order,
        scale=alt.Scale(scheme="blues"),
    ),
    row=alt.Row("Service:N", title=None),
    tooltip=["Service", "Stage", "Count"],
).properties(width=400, height=60, title="Baseline coverage funnel by exclusion tier")

funnel

alt.Chart(...)